# 70. RTX 3090에서 원판 검출 학습 일정 비교

원본 `yolo26n.pt`에서 시작한 두 실행의 `best.pt`를 비교합니다. 기존 일정은 최대 100에폭·patience 100, 새 일정은 최대 200에폭·patience 30입니다. 두 실행은 같은 선수별 분할과 학습 이미지 45장·검증 이미지 39장을 사용합니다. 최대 에폭을 바꾸면 학습률 감소 속도와 모자이크가 끝나는 시점도 달라집니다.

검증 선수 6명의 원본을 처음부터 처리하고, 시간 연결 후 동작 중 수동 박스가 있는 270장을 채점합니다. 프레임별 직접 검출도 같은 표에 표시합니다. 허브 세로 오차는 직접 찍은 허브와 검출 박스 중심의 y좌표 차이에 절댓값을 취한 평균입니다.

**실측 결과** 시간 연결 후 미탐이 17장에서 18장으로 늘었습니다. a03에서 복구한 4장과 b02에서 새로 놓친 5장을 아래에서 비교합니다.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT

experiment = ROOT / "outputs/plate_3090_schedule/20261126"
review = experiment / "paired_review"
control_path = experiment / "control_before_training.csv"
if control_path.exists():
    display(pd.read_csv(control_path).round(3))

### 두 일정의 미탐과 허브 오차

동작 중 미탐 17장 이하, a03 최장 연속 미탐 13프레임 이하, 공통 정탐에서 허브 세로 오차 증가 1px 이하일 때 새 학습 일정을 채택합니다. 바로 위 대조군 표의 허브 오차는 대조군이 맞힌 모든 직접 판독점에 해당합니다. 아래 비교 표에서는 두 모델이 함께 맞힌 직접 판독점에서 오차를 다시 계산합니다.

In [ ]:
ready = (review / "decision.json").is_file()
if ready:
    summary = pd.read_csv(review / "summary.csv")
    display(summary.rename(columns={
        "method": "후처리", "schedule": "최대 에폭/patience",
        "motion_misses": "동작 중 미탐(장)", "a03_longest": "a03 최장 연속 미탐(프레임)",
        "common_hubs": "공통 허브(장)", "hub_y_mae_px": "허브 세로 MAE(px)",
        "map50_95": "mAP50–95",
    }).round(3))
    decision = json.loads((review / "decision.json").read_text())
    print("새 일정 채택" if decision["accept_3090_schedule"] else "기존 일정 유지")
else:
    print("200/30 학습 및 짝지은 채점 결과를 기다리고 있습니다.")

### 어느 선수의 프레임을 새로 찾거나 놓쳤나

왼쪽 막대는 새 일정에서만 찾은 프레임, 오른쪽은 기존 일정에서만 찾은 프레임입니다. 아래 표에서 해당 프레임 번호를 확인합니다.

In [ ]:
if ready:
    detail = pd.read_csv(review / "frames.csv")
    linked = detail[detail.method.eq("Past link")]
    changes = linked[linked.change.isin(["복구", "손실"])]
    counts = pd.crosstab(changes.video_id, changes.change)
    counts = counts.reindex(index=sorted(linked.video_id.unique()),
                            columns=["복구", "손실"], fill_value=0)
    ax = counts.rename(columns={"복구": "Recovered", "손실": "Lost"}).plot.bar(
        color=["#176B87", "#AC5939"], rot=20, figsize=(8, 3))
    ax.set(xlabel="Validation video", ylabel="Frames",
           title="Newly recovered and lost detections during the lift")
    plt.tight_layout()
    plt.savefig(review / "recovered_lost.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(changes[["video_id", "frame_index", "change", "iou_base", "iou_trial"]].round(3))

### 복구와 손실 프레임에서 박스와 허브 보기

같은 원본을 두 열에 표시합니다. 초록색 점선은 수동 박스, 주황색 실선은 모델 박스입니다. 사람이 직접 표시한 허브는 흰색 십자로 나타냅니다. b02 35번에서 신뢰도와 IoU가 서로 다르게 변한 이유를 박스의 크기와 중심에서 확인합니다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.config import VIDEOS
from wlcv.data.frames import read_rgb

if ready:
    examples = [("a03_sn_f_01", 14), ("b02_sn_h_02", 35)]
    fig, axes = plt.subplots(2, 2, figsize=(8, 8), layout="constrained")
    for row, (video, frame) in enumerate(examples):
        rgb = read_rgb(VIDEOS / f"{video}.mp4", frame)
        item = linked[(linked.video_id == video) & (linked.frame_index == frame)].iloc[0]
        for col, (model, schedule) in enumerate([("base", "100/100"), ("trial", "200/30")]):
            ax = axes[row, col]
            ax.imshow(rgb)
            x, y = item.gt_x1, item.gt_y1
            w, h = item.gt_x2 - x, item.gt_y2 - y
            ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor="lime",
                                   linestyle="--", linewidth=2))
            pred_x, pred_y = item[f"x1_{model}"], item[f"y1_{model}"]
            label = "no box"
            if pd.notna(pred_x):
                ax.add_patch(Rectangle((pred_x, pred_y), item[f"x2_{model}"] - pred_x,
                                       item[f"y2_{model}"] - pred_y, fill=False,
                                       edgecolor="darkorange", linewidth=2))
                label = f"conf {item[f'confidence_{model}']:.3f} | IoU {item[f'iou_{model}']:.3f}"
            if item.visibility == "visible" and pd.notna(item.manual_y):
                ax.plot(item.manual_x, item.manual_y, "w+", markersize=12, markeredgewidth=2)
            ax.set(xlim=(max(0, x-w), min(rgb.shape[1], x+2*w)),
                   ylim=(min(rgb.shape[0], y+2*h), max(0, y-h)))
            ax.set_title(f"{video[:3]} #{frame} | {schedule}\n{label}")
            ax.axis("off")
    fig.savefig(review / "frame_examples.png", dpi=150)
    plt.show()